# Kaggle 2/2 - Train nnU-Net (GPU)

**Settings:** Accelerator = **GPU T4 x2** (or P100), Internet = **On**. Add notebook 1's output as an input (**Add Input -> Your Work**).
If this is a continuation of an earlier training run, also add *this* notebook's previous output as an input: checkpoints are restored automatically.

Run cells in order. For the long training cell use **Save Version -> Save & Run All (Commit)** so it keeps running with the browser closed.

In [ ]:
# 0. GPU check. Stop if no GPU is shown.
!nvidia-smi | head -n 15
import torch; print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

In [ ]:
# 1. Code + install. Do NOT pip install torch. Then install the 10-epoch-checkpoint trainers.
import os
if not os.path.exists('brats-nnunet'):
    !git clone https://github.com/Hidaayet/brats-nnunet
%cd brats-nnunet
!pip install -q nnunetv2 nibabel scipy pandas matplotlib tqdm
!python scripts/install_trainer.py

In [ ]:
# 2. Find the prepared data (and old checkpoints) in /kaggle/input and copy them to fast local disk
import os, shutil, pathlib
def find_dir(name, root='/kaggle/input'):
    for dp, dn, _ in os.walk(root):
        if name in dn: return pathlib.Path(dp) / name
    return None

BASE = pathlib.Path('/kaggle/temp/nnunet'); DS = 'Dataset501_BrainTumour'
pre_src = find_dir('nnUNet_preprocessed')
assert pre_src, 'Prepared data not found: add notebook 1 output as an input (Add Input -> Your Work).'
if not (BASE / 'nnUNet_preprocessed' / DS).exists():
    shutil.copytree(pre_src / DS, BASE / 'nnUNet_preprocessed' / DS)

res = pathlib.Path('/kaggle/working/nnUNet_results')
old = find_dir('nnUNet_results')
if old and not res.exists():
    shutil.copytree(old, res); print('Restored earlier checkpoints from', old)

(BASE / 'nnUNet_raw').mkdir(parents=True, exist_ok=True)
os.environ.update(BASE=str(BASE), RESULTS_DIR=str(res))
os.environ['TRAINER'] = 'nnUNetTrainer_100ep_s10'      # the ONE place to set the epoch count
CONFIG, FOLD = '3d_fullres', 0
!ls /kaggle/working/nnUNet_results 2>/dev/null | head

In [ ]:
# 3. OPTIONAL timing test (5 epochs, about 10-20 min): read the epoch time to choose the real epoch count.
!TRAINER=nnUNetTrainer_5epochs bash scripts/02_train_nnunet.sh $CONFIG $FOLD
!grep -h "Epoch time" /kaggle/working/nnUNet_results/Dataset501_BrainTumour/nnUNetTrainer_5epochs__nnUNetPlans__3d_fullres/fold_0/training_log_*.txt | tail -n 5

In [ ]:
# 4. The real training. 'timeout' stops it cleanly before the session limit so the output (checkpoints) is saved.
# Resume = just run this cell again in a new session (checkpoint found automatically).
!timeout 41000 bash scripts/02_train_nnunet.sh $CONFIG $FOLD
!du -sh /kaggle/working/nnUNet_results

In [ ]:
# 5. ONLY when training has finished (checkpoint_final.pth exists): predict the held-out patients and score them. Do this ONCE.
import glob
fin = glob.glob(f"/kaggle/working/nnUNet_results/{DS}/{os.environ['TRAINER']}__nnUNetPlans__{CONFIG}/fold_{FOLD}/checkpoint_final.pth")
assert fin, 'Training not finished yet: run cell 4 again (resumes).'
hold = find_dir('heldout')
raw = BASE / 'nnUNet_raw' / DS; raw.mkdir(parents=True, exist_ok=True)
for sub in ['imagesTs_heldout', 'labelsTs_heldout']:
    if not (raw / sub).exists(): shutil.copytree(hold / sub, raw / sub)
!bash scripts/03_predict_evaluate.sh $CONFIG $FOLD

In [ ]:
# 6. Statistics, failure analysis and one overlay figure
NAME = f'nnunet_{CONFIG}_fold{FOLD}'
!python -m src.failure_analysis --csv results/tables/{NAME}_per_case.csv --k 10
import pandas as pd
worst = pd.read_csv(f'results/tables/{NAME}_worst10.csv')['case'].iloc[0]
!python -m src.overlay --case $worst --images {BASE}/nnUNet_raw/{DS}/imagesTs_heldout --gt {BASE}/nnUNet_raw/{DS}/labelsTs_heldout --pred results/predictions/{NAME}
# Copy tables/figures out of the clone so they appear in the notebook output, then add them to GitHub by hand
!mkdir -p /kaggle/working/results_export && cp -r results/tables results/figures /kaggle/working/results_export/
!pip freeze | grep -iE "^(torch|nnunetv2|nibabel|numpy|scipy|pandas|batchgenerators)==" > /kaggle/working/results_export/environment.txt; cat /kaggle/working/results_export/environment.txt

## For the 2D baseline
Run notebook 1 again with `CONFIGS = '2d'`, then this notebook with `CONFIG = '2d'`. It uses the same frozen split. Then compare:
`python -m src.compare --a results/tables/nnunet_3d_fullres_fold0_per_case.csv --name-a 3D --b results/tables/nnunet_2d_fold0_per_case.csv --name-b 2D`